<a href="https://colab.research.google.com/github/veronicamarenco/EDC/blob/main/Marenco_NLP_Lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Lab 3 - Part 1: Text Visualization & Classical Representations
#Objectives:

#Visualize text data using bar charts, word clouds, and custom visualizations
#Implement Bag of Words (BoW) and TF-IDF representations
#Work with N-grams and build a simple next-word predictor
#Analyze real news data and interpret results


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re
import string

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from wordcloud import WordCloud, STOPWORDS
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print("Setup complete!")

In [ ]:
# Load the dataset
import pandas as pd

# Using the Hugging Face datasets library
from datasets import load_dataset

# Load the entire dataset
dataset = load_dataset("SetFit/20_newsgroups")

# Combine train and test splits, or use just train
df = pd.DataFrame(dataset['train'])

# If you want both train and test combined:
# df = pd.concat([
#     pd.DataFrame(dataset['train']),
#     pd.DataFrame(dataset['test'])
# ], ignore_index=True)


In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df['label_text'].value_counts())

In [ ]:
# View sample data
print("Sample document:")
print("="*50)
print(f"Label: {df.iloc[0]['label_text']}")
print(f"Text (first 500 chars): {df.iloc[0]['text'][:500]}...")

In [ ]:
# Exercise A.1: Select YOUR Categories

In [ ]:
# TODO: Choose YOUR 3 categories (this affects all your analysis!)
# YOUR CODE HERE
my_categories = ["sci.space", "comp.graphics", "misc.forsale"]  # Replace with your choices

# Filter the dataset
df_filtered = df[df['label_text'].isin(my_categories)].copy()
df_filtered = df_filtered.reset_index(drop=True)

print(f"Selected categories: {my_categories}")
print(f"Filtered dataset size: {len(df_filtered)}")
print(f"\nDistribution:")
print(df_filtered['label_text'].value_counts())

Written Question A.1 (Personal Interpretation)
Why did you choose these 3 specific categories? Explain your reasoning (at least 3 sentences).

Consider:

Are they related or completely different?
What do you expect to find in terms of vocabulary differences?
Why are they interesting to YOU?
YOUR ANSWER:

[Write your answer here - minimum 3 sentences]

Space and Astronomy have been strong interests of mine since I was a kid. I love learning new things as the technology continues to develop. I also love tech, I used to keep up with the latest tech gadgets, but now I'm learning more logic, coding and how machines learn.
Lastly, I chose misc.forsale because sometimes it's cool to go thrifting or see what other people have to sell.

I would argue that tech and space defintely have overlap but I would be surprised to find out what they have in common with the for sale category.

# Part B: Text Preprocessing **Function**

Before visualization, we need to clean our text data.


In [ ]:
# Example preprocessing function
# TODO: Complete the function as needed
def preprocess_text(text):
    """Basic text preprocessing."""
    # Lowercase
    text = text.lower()
    # Remove emails (pattern: word@word.word)
    text = re.sub(r'\S+@\S+', '', text)
    # Remove URLs (http/https/www)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove extra whitespace (multiple spaces, tabs, newlines)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Test
sample = "Hello! Check this: http://example.com. Email me at test@email.com. Price: $100."
print(f"Original: {sample}")
print(f"Cleaned:  {preprocess_text(sample)}")

# Exercise B.1: Improve the Preprocessing **Function**

The function above is basic. Improve it by adding:

Stop word removal
Lemmatization
Minimum word length filter (remove words with < 3 characters)

In [ ]:
import nltk

nltk.download('punkt_tab', quiet=True) # Explicitly download punkt_tab because I was getting an error message

# TODO: Complete this improved preprocessing function
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text_advanced(text):
    """
    Advanced text preprocessing with stop words removal and lemmatization.

    Args:
        text (str): Input text
    Returns:
        str: Preprocessed text
    """
    # YOUR CODE HERE
    # Step 1: Basic cleaning (lowercase, remove emails, URLs, numbers, punctuation)
    text = text.lower()
    text = re.sub(r'\S+@\S+', '', text)                    # Remove emails
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)    # Remove URLs
    text = re.sub(r'\d+', '', text)                        # Remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()               # Remove extra whitespace
    # Step 2: Tokenize
    tokens = word_tokenize(text)
    # Step 3: Remove stop words
    tokens = [token for token in tokens if token not in stop_words]

    # Step 4: Lemmatize
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    # Step 5: Remove short words (< 3 chars)
    tokens = [token for token in tokens if len(token) >= 3]
    # Step 6: Join back to string
    result = ' '.join(tokens)
    return result  # Corrected: return result instead of ""

# Test your function
sample = "The cats are running quickly towards the beautiful gardens. Email: test@mail.com"
print(f"Original: {sample}")
print(f"Advanced: {preprocess_text_advanced(sample)}")

In [ ]:
# Apply preprocessing to your filtered dataset
df_filtered['text_clean'] = df_filtered['text'].apply(preprocess_text_advanced)

# Show sample
print("Sample preprocessed document:")
print(df_filtered.iloc[0]['text_clean'][:300])

## Part C: Text Visualization
### C.1 Bar Chart: Top Words per Category

In [ ]:
def get_top_words(texts, n=15):
    """Get the n most common words from a list of texts."""
    all_words = ' '.join(texts).split()
    word_counts = Counter(all_words)
    return word_counts.most_common(n)

# Get top words for each category
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, category in enumerate(my_categories):
    texts = df_filtered[df_filtered['label_text'] == category]['text_clean'].tolist()
    top_words = get_top_words(texts, 15)

    words, counts = zip(*top_words)
    axes[idx].barh(words, counts, color=plt.cm.Set2(idx))
    axes[idx].set_title(f'Top 15 Words: {category}')
    axes[idx].invert_yaxis()
    axes[idx].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('top_words_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

Written Question C.1 (Personal Interpretation)
Look at your bar charts above and answer:

What words are UNIQUE to each category? (List at least 2 per category)
What words are SHARED across categories? Why do you think they appear in multiple categories?
Based ONLY on the top words, could you guess the topic of each category? Explain.
YOUR ANSWER:

Category 1 sci.space:

Unique words: ...

Category 2 comp.graphics:

Unique words: ...

Category 3 misc.forsale:

Unique words: ...

Shared words and explanation: ...

Topic guessing analysis: ...

In [ ]:
# Simple word cloud for each category
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

colors = ['Blues', 'Greens', 'Reds']

for idx, category in enumerate(my_categories):
    texts = df_filtered[df_filtered['label_text'] == category]['text_clean'].tolist()
    text_combined = ' '.join(texts)

    wordcloud = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap=colors[idx],
        max_words=100,
        min_font_size=10
    ).generate(text_combined)

    axes[idx].imshow(wordcloud, interpolation='bilinear')
    axes[idx].set_title(f'Word Cloud: {category}', fontsize=14)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('wordclouds_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:

# TODO: Create a custom word cloud with a mask
# Choose ONE of your categories for this visualization

from PIL import Image

# Create a circular mask
def create_circle_mask(size=400):
    x = np.arange(0, size)
    y = np.arange(0, size)
    cx, cy = size // 2, size // 2
    r = size // 2 - 10
    mask = np.zeros((size, size), dtype=np.uint8)
    for i in x:
        for j in y:
            if (i - cx)**2 + (j - cy)**2 <= r**2:
                mask[j, i] = 255
    return mask

circle_mask = create_circle_mask(400)

# TODO: Create a custom word cloud with a mask
# Choose ONE of your categories for this visualization

# YOUR CODE HERE
selected_category = "sci.space"  # Choose one of your categories

# Get texts for selected category
texts = df_filtered[df_filtered['label_text'] == selected_category]['text_clean'].tolist()
text_combined = ' '.join(texts)

# Create word cloud with mask
# Hint: Use the mask parameter in WordCloud()
wordcloud_masked = WordCloud(
        width=800,
        height=400,
        background_color='white',
        mask=circle_mask, # Use the created mask
        max_words=100,
        min_font_size=10
    ).generate(text_combined)

# Display
plt.figure(figsize=(10, 10))
plt.imshow(wordcloud_masked, interpolation='bilinear') # Display wordcloud_masked
plt.title(f'Custom Word Cloud: {selected_category}')
plt.axis('off')
plt.savefig('custom_wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()

# Exercise D.1: Create BoW for Your **Dataset**

In [ ]:
# Example: Simple Bag of Words
sample_docs = [
    "I love machine learning",
    "Machine learning is great",
    "I love deep learning too"
]

# Create BoW vectorizer
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform(sample_docs)

# Show vocabulary
print("Vocabulary:", bow_vectorizer.get_feature_names_out())
print("\nBoW Matrix (dense):")
print(bow_matrix.toarray())

# As DataFrame
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow_vectorizer.get_feature_names_out())
print("\nAs DataFrame:")
bow_df

In [ ]:
#Exercise D.1: Create BoW for Your Dataset

In [ ]:
# TODO: Create a Bag of Words representation for your filtered dataset
# Use parameters: max_features=1000, min_df=5, max_df=0.95

bow_vectorizer_full = CountVectorizer(
    max_features=1000,
    min_df=5,
    max_df=0.95
)

# Fit and transform on your cleaned texts
bow_matrix_full = None
bow_matrix_full = bow_vectorizer_full.fit_transform(texts)

print(f"BoW Matrix shape: {bow_matrix_full.shape}")
print(f"Vocabulary size: {len(bow_vectorizer_full.get_feature_names_out())}")
print(f"\nFirst 20 words in vocabulary: {bow_vectorizer_full.get_feature_names_out()[:20]}")

# Exercise D.2: Document Similarity with BoW

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Compute cosine similarity between documents
similarity_matrix = cosine_similarity(bow_matrix_full)

print(f"Similarity matrix shape: {similarity_matrix.shape}")

# --- Most SIMILAR pair (excluding self-similarity on diagonal) ---
sim_copy = similarity_matrix.copy()
np.fill_diagonal(sim_copy, -1)  # mask self-similarity

most_similar_idx = np.unravel_index(np.argmax(sim_copy), sim_copy.shape)
doc_a, doc_b = most_similar_idx
print(f"\n2 Most SIMILAR documents:")
print(f"  Doc {doc_a} & Doc {doc_b} — similarity: {similarity_matrix[doc_a, doc_b]:.4f}")

# --- Most DIFFERENT pair (lowest cosine similarity) ---
most_different_idx = np.unravel_index(np.argmin(sim_copy), sim_copy.shape)
doc_c, doc_d = most_different_idx
print(f"\n2 Most DIFFERENT documents:")
print(f"  Doc {doc_c} & Doc {doc_d} — similarity: {similarity_matrix[doc_c, doc_d]:.4f}")

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Create a Bag of Words representation for your filtered dataset
bow_vectorizer_full = CountVectorizer(
    max_features=1000,
    min_df=5,
    max_df=0.95
)

# Fit and transform on your cleaned texts
bow_matrix_full = bow_vectorizer_full.fit_transform(df_filtered['text_clean'])

print(f"BoW Matrix shape: {bow_matrix_full.shape}")
print(f"Vocabulary size: {len(bow_vectorizer_full.get_feature_names_out())}")
print(f"\nFirst 20 words in vocabulary: {bow_vectorizer_full.get_feature_names_out()[:20]}")

Written Question D.1 (Personal Interpretation)
Look at the 2 most similar documents you found:

Are they from the same category or different categories?
Read the original texts (first 200 characters). What makes them similar?
Is the BoW similarity measure meaningful here? Why or why not?

ANSWER

In [ ]:
# Show the similar documents for your analysis
print("Document 1 (first 300 chars):")
print(df_filtered.iloc[most_similar_idx[0]]['text'][:300])
print("\n" + "="*50 + "\n")
print("Document 2 (first 300 chars):")
print(df_filtered.iloc[most_similar_idx[1]]['text'][:300])